# sokudan quickstart (v0.2.1)

[sokudan](https://github.com/hiroki-abe-58/sokudan) is a Japanese System One decision model: send text and typed questions, get typed answers with probabilities, without generating a single token.
Model: [`GeneLab/sokudan-ja-310m`](https://huggingface.co/GeneLab/sokudan-ja-310m) (314.6M parameters). **A CPU runtime is enough.**

This notebook: install from PyPI → the three question types (`choice`, `score`, `noul`) → bool calibration on and off → `sokudan serve` inside the notebook, called once with `curl`.

日本語: 業務文と型付きの質問を渡すと、テキストを生成せずに型付きの答えと確率を返すモデルです。無料の CPU ランタイムで動きます。

## 1. Install

`sokudan` supports Python 3.11–3.13 (from 0.3.0). Colab already has torch; `pip` keeps it.

In [ ]:
import subprocess
import sys

cmd = [sys.executable, "-m", "pip", "install", "-q", "sokudan[serve]>=0.3.0"]
print(" ".join(cmd[1:]))
r = subprocess.run(cmd, capture_output=True, text=True)
if r.returncode:
    raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
print("installed")

## 2. Three question types

The state is passed **as a string** (a dict is rendered as `key: value` lines, which is not the training input).
The first call downloads the weights (about 1.3 GB).

In [ ]:
import json

import sokudan

print("sokudan", sokudan.__version__)
agent = sokudan.load("GeneLab/sokudan-ja-310m")
print("device:", agent.device)

state = "先月の請求で同じ金額が二回引き落とされています。至急ご確認ください。"
questions = {
    "department": {"type": "choice",
                   "instructions": "この問い合わせはどの部署が担当すべきか",
                   "criteria": {"請求": "支払い・返金", "技術": "不具合・障害",
                                "営業": "料金・新規契約", "その他": "上記以外"}},
    "urgency":    {"type": "score",
                   "instructions": "この依頼の緊急度は",
                   "criteria": ["急がない", "早めに", "業務が止まっている"]},
    "churn":      {"type": "noul",
                   "instructions": "解約を示唆しているか"},
}
result = agent.predict(state, questions)
print(result["answers"]["department"]["choice"])
print(json.dumps(result["answers"], ensure_ascii=False, indent=1))

## 3. Calibration on and off

v0.2.1 applies one temperature to `noul`/`bool` answers by default (the `calibration.json` shipped with the weights). `choice` and `score` stay raw.
`temperatures=None` gives the raw probabilities (as in v0.2). The temperature is monotonic: the ranking and the 0.5 decision do not change; P(true) moves toward 0.5.
On `bench_ja` the bool ECE went 0.181 → 0.105 ([docs/calibration.md](https://github.com/hiroki-abe-58/sokudan/blob/main/docs/calibration.md)).

In [ ]:
raw = sokudan.load("GeneLab/sokudan-ja-310m", temperatures=None)

churn = {"churn": questions["churn"]}
states = [
    state,
    "他社のほうが安いので、今月いっぱいで契約を終わりにしたいと考えています。",
    "新しいプランの料金表を送っていただけますか。",
]
print(f"{'P(true) calibrated':>20} {'raw':>8}  state")
for s in states:
    on = agent.predict(s, churn)
    off = raw.predict(s, churn)
    print(f"{on['answers']['churn']['noul']:>20.4f} {off['answers']['churn']['noul']:>8.4f}  {s}")
print("calibrated:", on["calibrated"], on["calibrated_answers"], "/ raw:", off["calibrated"])

del raw, agent  # free memory before starting the server

## 4. `sokudan serve` in the notebook

`sokudan serve` answers `POST /v1/systemone` in the same wire format as TypeSafe's public API reference ([docs/serving.md](https://github.com/hiroki-abe-58/sokudan/blob/main/docs/serving.md)). It is started in the background here and called once with `curl`.

In [ ]:
import time
import urllib.request

server = subprocess.Popen([sys.executable, "-m", "sokudan.cli", "serve", "--port", "8000"],
                          stdout=open("serve.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        health = json.load(urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2))
        break
    except OSError:
        if server.poll() is not None:
            raise RuntimeError(open("serve.log").read()) from None
        time.sleep(1)
else:
    raise RuntimeError("server did not start; see serve.log")
print({k: health[k] for k in ("status", "device", "sokudan_version", "calibrated")})

In [ ]:
# the request body is written to a file (UTF-8), then sent with curl
with open("req.json", "w", encoding="utf-8") as f:
    json.dump({"state": state, "questions": questions}, f, ensure_ascii=False)

In [ ]:
url = "http://127.0.0.1:8000/v1/systemone"
!curl -s $url -H "content-type: application/json" --data-binary @req.json -o resp.json

In [ ]:
with open("resp.json", encoding="utf-8") as f:
    resp = json.load(f)
print(json.dumps(resp, ensure_ascii=False, indent=1))
assert {a["type"] for a in resp["answers"].values()} == {"choice", "score", "noul"}

In [ ]:
server.terminate()
server.wait(timeout=30)
print("server stopped:", server.returncode)

## More

- Code, metrics and limits: [github.com/hiroki-abe-58/sokudan](https://github.com/hiroki-abe-58/sokudan) (Japanese: [README_ja.md](https://github.com/hiroki-abe-58/sokudan/blob/main/README_ja.md))
- Model card: [GeneLab/sokudan-ja-310m](https://huggingface.co/GeneLab/sokudan-ja-310m)
- PyPI: [sokudan](https://pypi.org/project/sokudan/)